# 00 · JWST f150w Noise Characterization

Measure the sky background noise level from actual JWST f150w cutouts and fit a
lognormal distribution to replace the DECaLS-calibrated parameters in `GaussianNoise`.

**Sections:**
1. Load a sample of cutouts & measure per-image sky σ
2. Inspect pixel value scale (units)
3. Fit lognormal to the σ distribution
4. Sanity-check: simulate augmented noise and verify coverage
5. Summary — new parameters to paste into `augmentations.py`

In [ ]:
import os, glob
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.stats import lognorm
from astropy.stats import sigma_clipped_stats
from tqdm import tqdm

DATA_ROOT  = '/u/yacheng/projects/ssl_outthere/images/jwst/f150w'
N_SAMPLES  = 2000   # cutouts to measure (increase for better statistics)
SEED       = 42
DEG_TO_PIX = 3600 * 1000 / 30   # 1 degree → pixels at 30 mas/pix
rng = np.random.default_rng(SEED)

h5_files = sorted(glob.glob(os.path.join(DATA_ROOT, '*.h5')))
print(f'Found {len(h5_files)} HDF5 files')

## 1 · Measure per-cutout sky σ

**Why corners, not annulus:**
The annulus approach (masking central source, sigma-clipping the ring) is contaminated
by bright/large galaxies whose outer envelopes extend far from the centre.
This produces a **bimodal** distribution — the lower mode is true sky noise, the upper
mode is source-contaminated residuals.

Instead we use **corner pixels** (e.g. a 5×5 block at each of the 4 corners), which
are the farthest from the source and are sky-dominated for virtually all objects in the catalog.

- Record `sky_sigma`, median pixel value, and p95 pixel value (for unit inspection)

In [ ]:
CORNER_SIZE = 5   # pixels per side; 4 corners × 5×5 = 100 pixels total

sky_sigmas   = []
pix_medians  = []
pix_p95      = []

# Build an index of all valid cutouts across files
index = []   # (filepath, local_index)
for fp in h5_files:
    with h5py.File(fp, 'r') as f:
        n = f['image'].shape[0]
        for li in range(n):
            index.append((fp, li))

print(f'Total index size: {len(index)}')
chosen = rng.choice(len(index), size=min(N_SAMPLES, len(index)), replace=False)
print(f'Sampling {len(chosen)} cutouts...')

open_files = {}
try:
    for i in tqdm(chosen, desc='Measuring sky σ (corners)'):
        fp, li = index[i]
        if fp not in open_files:
            open_files[fp] = h5py.File(fp, 'r')
        img = open_files[fp]['image'][li].astype('float64')   # (H, W)
        H, W = img.shape

        c = min(CORNER_SIZE, H // 4, W // 4)   # safety: never take more than 25% side
        if c < 2:
            continue

        corners = np.concatenate([
            img[:c,  :c ].ravel(),   # top-left
            img[:c,  -c:].ravel(),   # top-right
            img[-c:, :c ].ravel(),   # bottom-left
            img[-c:, -c:].ravel(),   # bottom-right
        ])

        _, med, std = sigma_clipped_stats(corners, sigma=3.0, maxiters=5)
        sky_sigmas.append(std)
        pix_medians.append(med)
        pix_p95.append(float(np.percentile(img, 95)))
finally:
    for f in open_files.values():
        f.close()

sky_sigmas  = np.array(sky_sigmas)
pix_medians = np.array(pix_medians)
pix_p95     = np.array(pix_p95)

print(f'\nMeasured {len(sky_sigmas)} cutouts')
print(f'sky_sigma  — min={sky_sigmas.min():.4e}  p05={np.percentile(sky_sigmas,5):.4e}  '
      f'median={np.median(sky_sigmas):.4e}  p95={np.percentile(sky_sigmas,95):.4e}  '
      f'max={sky_sigmas.max():.4e}')
print(f'pix_median — median={np.median(pix_medians):.4e}  (→ should be near 0 if background-subtracted)')
print(f'Units hint : median sky_sigma = {np.median(sky_sigmas):.3e}  '
      f'(MJy/sr if ~0.01–0.05, nJy if ~10–100)')

## 2 · Pixel value scale — identify units

| Typical median value | Likely unit |
|---|---|
| ~1e-2 – 1 | MJy/sr (native JWST pipeline) |
| ~1 – 1e4 | nJy or counts |
| > 1e4 | Raw ADU (unlikely for mosaics) |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# — Units check —
axes[0].hist(pix_medians, bins=80, color='steelblue', edgecolor='none')
axes[0].set_xlabel('Corner median pixel value')
axes[0].set_ylabel('Count')
axes[0].set_title('Corner median (sky level)\n→ near 0 = background subtracted')
axes[0].axvline(np.median(pix_medians), color='red', lw=1.5, ls='--',
                label=f'median={np.median(pix_medians):.3e}')
axes[0].axvline(0, color='gray', lw=1, ls=':')
axes[0].legend(fontsize=9)

# — sky sigma linear scale —
axes[1].hist(sky_sigmas, bins=80, color='seagreen', edgecolor='none', alpha=0.8)
axes[1].set_xlabel('Sky σ (corner pixels)')
axes[1].set_ylabel('Count')
axes[1].set_title('Sky σ distribution (linear)\nshould be unimodal now')
for p, ls in zip([5, 50, 95], [':', '--', ':']):
    v = np.percentile(sky_sigmas, p)
    axes[1].axvline(v, color='red', lw=1.2, ls=ls, label=f'p{p}={v:.3e}')
axes[1].legend(fontsize=8)

# — sky sigma log scale —
log_s = np.log10(sky_sigmas + 1e-12)
axes[2].hist(log_s, bins=80, color='darkorange', edgecolor='none', alpha=0.8)
axes[2].set_xlabel('log₁₀(sky σ)')
axes[2].set_title('Sky σ on log scale')
axes[2].axvline(np.log10(np.median(sky_sigmas)), color='red', lw=1.5, ls='--',
                label=f'median={np.median(sky_sigmas):.3e}')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('sky_sigma_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Pixel units hint:')
print(f'  median background level : {np.median(pix_medians):.4e}')
print(f'  median sky sigma        : {np.median(sky_sigmas):.4e}')
print(f'  → If sky_sigma ~ 0.01-0.05 : MJy/sr  (JWST native)')
print(f'  → If sky_sigma ~ 10-100    : nJy')

## 3 · Fit lognormal to the sky σ distribution

The existing `GaussianNoise` draws noise sigma from a lognormal as:
```python
sigma = np.random.lognormal(sigma_dist, shape_dist) + loc_dist
# where sigma_dist = log(scale_dist)
```
This is equivalent to `scipy.stats.lognorm(s=shape, loc=loc, scale=scale)`
with `sigma_dist = log(scale)`.  We fit these three parameters below.

In [ ]:
# Remove obvious outliers before fitting (keep central 99%)
lo, hi = np.percentile(sky_sigmas, [0.5, 99.5])
fit_data = sky_sigmas[(sky_sigmas >= lo) & (sky_sigmas <= hi)]
print(f'Fitting on {len(fit_data)} samples (clipped outliers outside [{lo:.3e}, {hi:.3e}])')

# scipy lognorm.fit: s=shape, loc=loc, scale=scale
shape, loc, scale = lognorm.fit(fit_data, floc=0)  # fix loc=0 first
print(f'\nFit with floc=0:')
print(f'  shape = {shape:.6f}')
print(f'  loc   = {loc:.6e}')
print(f'  scale = {scale:.6e}')
print(f'  → sigma_dist (log scale) = {np.log(scale):.6f}')

# Also try free loc
shape_fl, loc_fl, scale_fl = lognorm.fit(fit_data)
print(f'\nFit with free loc:')
print(f'  shape = {shape_fl:.6f}')
print(f'  loc   = {loc_fl:.6e}')
print(f'  scale = {scale_fl:.6e}')
print(f'  → sigma_dist (log scale) = {np.log(scale_fl):.6f}')

# Percentiles for noise_ch_min / noise_ch_max
p05, p95 = np.percentile(sky_sigmas, [5, 95])
print(f'\n5th  percentile (→ noise_ch_min) : {p05:.6e}')
print(f'95th percentile (→ noise_ch_max) : {p95:.6e}')

# ── Plot: histogram + fitted lognormal ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(fit_data, bins=80, density=True, color='steelblue', alpha=0.6, label='measured sky σ')

x = np.linspace(fit_data.min(), fit_data.max(), 500)
ax.plot(x, lognorm.pdf(x, shape, loc, scale),   'r-',  lw=2, label=f'lognorm (floc=0)')
ax.plot(x, lognorm.pdf(x, shape_fl, loc_fl, scale_fl), 'g--', lw=2, label=f'lognorm (free loc)')
ax.axvline(p05, color='gray', ls=':', lw=1.5, label=f'p05={p05:.3e}')
ax.axvline(p95, color='gray', ls='--', lw=1.5, label=f'p95={p95:.3e}')
ax.set_xlabel('Sky σ (pixel units)')
ax.set_ylabel('Density')
ax.set_title('JWST f150w sky noise distribution + lognormal fit')
ax.legend()
plt.tight_layout()
plt.show()

## 4 · Sanity check — simulate GaussianNoise draws

Verify that the `sigma_augment` values produced by the fitted parameters
cover the measured `sky_sigma` range, and that they are not catastrophically
over- or under-estimating the noise.

We simulate the exact same draw used in `GaussianNoise.__call__`.

In [ ]:
# Use the free-loc fit (generally better)
s_fit  = shape_fl
l_fit  = loc_fl
sc_fit = scale_fl
sigma_dist_new = np.log(sc_fit)

N_SIM = 5000
# Arrays are 3-element in GaussianNoise, but all identical for single-band
_shape = np.array([s_fit,  s_fit,  s_fit ])
_loc   = np.array([l_fit,  l_fit,  l_fit ])
_sdist = np.array([sigma_dist_new, sigma_dist_new, sigma_dist_new])

sim_sigma_augment = []
for _ in range(N_SIM):
    sigma_true  = np.random.lognormal(_sdist, _shape) + _loc
    sigma_final = np.random.lognormal(_sdist, _shape) + _loc
    sigma_aug_sq = sigma_final**2 - sigma_true**2
    sigma_aug_sq[sigma_aug_sq < 0] = 0
    sim_sigma_augment.append(np.sqrt(sigma_aug_sq[0]))  # channel 0

sim_aug = np.array(sim_sigma_augment)
nonzero = sim_aug[sim_aug > 0]
print(f'Fraction of draws with nonzero augment: {len(nonzero)/N_SIM:.2%}')
print(f'Nonzero aug sigma — median={np.median(nonzero):.4e}  p95={np.percentile(nonzero,95):.4e}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sky_sigmas, bins=80, density=True, color='steelblue', alpha=0.5, label='measured sky σ')
if len(nonzero) > 0:
    ax.hist(nonzero, bins=80, density=True, color='tomato', alpha=0.5, label='simulated σ_augment (nonzero)')
ax.set_xlabel('σ (pixel units)')
ax.set_title('Measured sky σ vs simulated augmentation σ')
ax.set_xlim(0, 0.2)
ax.legend()
plt.tight_layout()
plt.show()

## 5 · Summary — paste these into `augmentations.py`

Replace the three DECaLS arrays in `GaussianNoise.__init__` with the values printed below.
Since the data is **single-band** (channel is broadcast 3× identically), all channels
use the same parameters.

In [ ]:
print('=' * 60)
print('New GaussianNoise parameters for JWST f150w')
print('=' * 60)
print(f'# Fitted to JWST COSMOS-Web f150w  (N={len(sky_sigmas)} cutouts)')
print(f'self.shape_dist = np.array([{s_fit:.7f}, {s_fit:.7f}, {s_fit:.7f}])')
print(f'self.loc_dist   = np.array([{l_fit:.7e}, {l_fit:.7e}, {l_fit:.7e}])')
print(f'self.scale_dist = np.array([{sc_fit:.7e}, {sc_fit:.7e}, {sc_fit:.7e}])')
print(f'self.sigma_dist = np.log(self.scale_dist)')
print()
print(f'self.noise_ch_min = np.array([{p05:.7e}, {p05:.7e}, {p05:.7e}])')
print(f'self.noise_ch_max = np.array([{p95:.7e}, {p95:.7e}, {p95:.7e}])')
print('=' * 60)
print()
print('Old DECaLS values (for reference):')
print('  shape_dist = [0.2264926, 0.2431146, 0.1334844]')
print('  loc_dist   = [-0.0006735, -0.0023663, -0.0143416]')
print('  scale_dist = [0.0037602, 0.0067417, 0.0260779]')
print('  noise_ch_min = [0.001094, 0.001094, 0.001094]')
print('  noise_ch_max = [0.013, 0.018, 0.061]')

## Next steps

1. Copy the printed values into `encoder_image/astrodino/train/data/augmentations.py`
   (the `GaussianNoise.__init__` block, lines ~267–276)
2. Visually spot-check: apply `GaussianNoise(im_dim=64)` to 10 random cutouts
   and confirm the added noise looks plausible relative to the original images
3. Restart training and verify no NaN loss on first few iterations

---

## 6 · Ground-truth sky σ: direct statistics on mosaic background pixels


In [ ]:
import gc
from astropy.io import fits
from astropy.stats import mad_std
from scipy.ndimage import binary_dilation

FITS_ROOT  = '/u/yacheng/projects/ssl_outthere/images/cosmos_2025'
SCI_TMPL   = os.path.join(FITS_ROOT, 'f150w',
             'mosaic_nircam_f150w_COSMOS-Web_30mas_{tile}_v1.0_sci.fits')
SEGM_TMPL  = os.path.join(FITS_ROOT, 'segmentation_maps',
             'detection_chi2pos_SWLW_{tile}_segmap_v1.3.fits.gz')

TILES_TO_USE    = ['A1' , 'A2', 'A3', 'A4', 'A6', 'A7', 'A8', 'A9', 'B1', 'B2', 'B3', 'B4', 'B6', 'B7', 'B8', 'B9',]
N_DILATE_MOSAIC = 10

tile_sigmas = {}

for tile in TILES_TO_USE:
    print(f'Tile {tile}...', end=' ', flush=True)

    # segmap: gzip → load fully → bool mask
    with fits.open(SEGM_TMPL.format(tile=tile)) as hdul:
        bg_mask = ~binary_dilation(hdul[0].data > 0, iterations=N_DILATE_MOSAIC)
    gc.collect()

    # sci: one sequential read → boolean index (much faster than random memmap seeks)
    with fits.open(SCI_TMPL.format(tile=tile)) as hdul:
        pixels = hdul[0].data[bg_mask].astype(np.float32)
    del bg_mask;  gc.collect()

    pixels = pixels[np.isfinite(pixels)]
    sigma  = mad_std(pixels)
    tile_sigmas[tile] = sigma
    del pixels;  gc.collect()
    print(f'σ_sky = {sigma:.5e} MJy/sr')

all_sigma = np.array(list(tile_sigmas.values()))
print(f'\nMean σ_sky : {all_sigma.mean():.5e} MJy/sr')
print(f'Tile variation: {all_sigma.std()/all_sigma.mean()*100:.1f}%  '
      f'range [{all_sigma.min():.5e}, {all_sigma.max():.5e}]')

In [ ]:
sigma_sky = all_sigma.mean()

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 1, figsize=(12, 4))

axes.bar(list(tile_sigmas.keys()), list(tile_sigmas.values()), color='steelblue', alpha=0.7)
axes.axhline(sigma_sky, color='red', ls='--', lw=1.5, label=f'mean={sigma_sky:.4e}')
axes.set_ylabel('σ_sky (MJy/sr)')
axes.set_title('Per-tile σ_sky (zoomed)')
axes.set_ylim(sigma_sky * 0.8, sigma_sky * 1.2)
axes.legend()

plt.tight_layout()
plt.show()

# ── Augmentation parameters ───────────────────────────────────────────────────
# uniform=True mode:
#   sigma_true  ~ tight lognormal around σ_sky  (current image noise)
#   sigma_final ~ uniform(0, k*σ_sky)           (target noise)
#   sigma_augment = sqrt(max(sigma_final² - sigma_true², 0))
#
# ~1/k of draws: sigma_final < sigma_true → no noise added (clean view)
# rest: sigma_augment ∈ (0, sqrt(k²-1)*σ_sky)

for k_max in [2, 3, 5]:
    aug_max = np.sqrt(k_max**2 - 1) * sigma_sky
    frac_noisy = 1 - 1/k_max
    print(f'k={k_max}:  noise_ch_max={k_max*sigma_sky:.4e}  '
          f'→ max sigma_augment={aug_max:.4e}  '
          f'({frac_noisy:.0%} of draws add noise)')

print()
K = 3
print(f'=== Recommended parameters (k={K}) ===')
print(f'uniform      = True')
print(f'noise_ch_min = np.array([0., 0., 0.])')
print(f'noise_ch_max = np.array([{K*sigma_sky:.6e}]*3)   # {K}× σ_sky')
print(f'shape_dist   = np.array([0.15, 0.15, 0.15])       # narrow → sigma_true ≈ σ_sky')
print(f'loc_dist     = np.array([0., 0., 0.])')
print(f'scale_dist   = np.array([{sigma_sky:.6e}]*3)      # lognormal median = σ_sky')
print(f'sigma_dist   = np.log(scale_dist)')

In [ ]:
# ── Bootstrap lognormal fit from background pixels ────────────────────────────
# 思路：从背景像素里反复随机抽取大小 = 训练 crop 面积 的子集，
# 每次算 mad_std → 得到 σ 的 bootstrap 分布 → 拟合 lognormal
#
# 这个分布的含义：如果随机取一个 64×64 背景 patch，σ 会落在哪个范围
# 这正好对应 GaussianNoise 里 sigma_true 应该建模的东西

BOOTSTRAP_TILE   = 'A5'                   # 用一个 tile 就够
CROP_AREA        = 64 * 64                # 训练 crop 的像素数
N_BOOTSTRAP      = 5000                   # bootstrap 次数
rng_bs = np.random.default_rng(0)

print(f'Loading background pixels for tile {BOOTSTRAP_TILE}...')
with fits.open(SEGM_TMPL.format(tile=BOOTSTRAP_TILE)) as hdul:
    bg_mask_bs = ~binary_dilation(hdul[0].data > 0, iterations=N_DILATE_MOSAIC)
with fits.open(SCI_TMPL.format(tile=BOOTSTRAP_TILE)) as hdul:
    bg_pixels_bs = hdul[0].data[bg_mask_bs].astype(np.float32)
del bg_mask_bs;  gc.collect()
bg_pixels_bs = bg_pixels_bs[np.isfinite(bg_pixels_bs)]
print(f'  {len(bg_pixels_bs):,} background pixels available')

# Bootstrap
boot_sigmas = np.array([
    mad_std(rng_bs.choice(bg_pixels_bs, size=CROP_AREA, replace=False))
    for _ in range(N_BOOTSTRAP)
])
print(f'Bootstrap σ — median={np.median(boot_sigmas):.4e}  '
      f'std={np.std(boot_sigmas):.4e}  '
      f'cv={np.std(boot_sigmas)/np.median(boot_sigmas)*100:.1f}%')

# Lognormal fit
s_bs, loc_bs, sc_bs = lognorm.fit(boot_sigmas)
print(f'\nLognormal fit (free loc):')
print(f'  shape = {s_bs:.6f}')
print(f'  loc   = {loc_bs:.6e}')
print(f'  scale = {sc_bs:.6e}  →  log(scale) = {np.log(sc_bs):.6f}')

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(boot_sigmas, bins=60, density=True, color='tomato', alpha=0.7,
        label=f'bootstrap σ  (N={N_BOOTSTRAP}, crop={CROP_AREA}px)')
x = np.linspace(boot_sigmas.min(), boot_sigmas.max(), 300)
ax.plot(x, lognorm.pdf(x, s_bs, loc_bs, sc_bs), 'k-', lw=2, label='lognorm fit')
ax.axvline(sigma_sky, color='steelblue', lw=1.5, ls='--', label=f'global σ_sky={sigma_sky:.4e}')
ax.set_xlabel('σ (MJy/sr)')
ax.set_title(f'Bootstrap σ distribution at crop scale ({int(CROP_AREA**0.5)}×{int(CROP_AREA**0.5)} px)')
ax.legend()
plt.tight_layout()
plt.show()

del bg_pixels_bs;  gc.collect()

---

## 7 · Multi-tile patch sampling → lognormal parameters for `GaussianNoise`



In [ ]:
import gc
from astropy.io import fits
from astropy.stats import mad_std
from scipy.ndimage import binary_dilation
from scipy.stats import lognorm
import numpy as np
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────────────────────
FITS_ROOT  = '/u/yacheng/projects/ssl_outthere/images/cosmos_2025'
SCI_TMPL   = os.path.join(FITS_ROOT, 'f150w',
             'mosaic_nircam_f150w_COSMOS-Web_30mas_{tile}_v1.0_sci.fits')
SEGM_TMPL  = os.path.join(FITS_ROOT, 'segmentation_maps',
             'detection_chi2pos_SWLW_{tile}_segmap_v1.3.fits.gz')

TILES       = ['A1','A2','A3','A4','A6','A7','A8','A9',
               'B1','B2','B3','B4','B6','B7','B8','B9']
N_DILATE    = 10       # dilation radius around sources (pixels)
CROP_AREA   = 64 * 64  # pixels per patch  (matches global_crops_size)
N_PER_TILE  = 1000     # patches to sample per tile
SEED        = 42

rng = np.random.default_rng(SEED)
all_patch_sigmas = []   # collects one σ per patch, across all tiles

# ── Loop over tiles ───────────────────────────────────────────────────────────
for tile in TILES:
    print(f'Tile {tile}...', end=' ', flush=True)

    # 1. Load segmap + dilate → background boolean mask
    with fits.open(SEGM_TMPL.format(tile=tile)) as hdul:
        bg_mask = ~binary_dilation(hdul[0].data > 0, iterations=N_DILATE)

    # 2. Load science image and extract background pixels
    with fits.open(SCI_TMPL.format(tile=tile)) as hdul:
        pixels = hdul[0].data[bg_mask].astype(np.float32)
    del bg_mask; gc.collect()

    pixels = pixels[np.isfinite(pixels)]
    print(f'{len(pixels):,} background pixels — ', end='', flush=True)

    # 3. Sample N_PER_TILE patches; each patch = CROP_AREA random pixels
    if len(pixels) < CROP_AREA:
        print('too few pixels, skipping')
        del pixels; gc.collect()
        continue

    patch_sigmas = np.array([
        mad_std(rng.choice(pixels, size=CROP_AREA, replace=False))
        for _ in range(N_PER_TILE)
    ])
    all_patch_sigmas.append(patch_sigmas)
    print(f'σ median={np.median(patch_sigmas):.4e}  cv={patch_sigmas.std()/patch_sigmas.mean()*100:.1f}%')
    del pixels; gc.collect()

all_patch_sigmas = np.concatenate(all_patch_sigmas)
print(f'\nTotal patches: {len(all_patch_sigmas):,}')
print(f'Global σ — median={np.median(all_patch_sigmas):.4e}  '
      f'p05={np.percentile(all_patch_sigmas,5):.4e}  '
      f'p95={np.percentile(all_patch_sigmas,95):.4e}')


In [ ]:
# ── Fit lognormal to aggregated patch σ values ────────────────────────────────
shape_fit, loc_fit, scale_fit = lognorm.fit(all_patch_sigmas)
p05, p95 = np.percentile(all_patch_sigmas, [5, 95])

print(f'Lognormal fit (free loc):')
print(f'  shape = {shape_fit:.7f}')
print(f'  loc   = {loc_fit:.7e}')
print(f'  scale = {scale_fit:.7e}')
print(f'  log(scale) = {np.log(scale_fit):.7f}')
print(f'  p05  = {p05:.6e}')
print(f'  p95  = {p95:.6e}')

# ── Plot histogram + fit ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

x = np.linspace(all_patch_sigmas.min(), all_patch_sigmas.max(), 500)

axes[0].hist(all_patch_sigmas, bins=80, density=True,
             color='steelblue', alpha=0.7, label=f'patch σ  (N={len(all_patch_sigmas):,})')
axes[0].plot(x, lognorm.pdf(x, shape_fit, loc_fit, scale_fit),
             'r-', lw=2, label='lognorm fit (free loc)')
axes[0].axvline(p05, color='gray', ls=':', lw=1.5, label=f'p05={p05:.3e}')
axes[0].axvline(p95, color='gray', ls='--', lw=1.5, label=f'p95={p95:.3e}')
axes[0].set_xlabel('σ per patch (MJy/sr)')
axes[0].set_ylabel('Density')
axes[0].set_title(f'Patch σ distribution — all tiles\n({len(TILES)} tiles × {N_PER_TILE} patches, crop={int(CROP_AREA**0.5)}×{int(CROP_AREA**0.5)} px)')
axes[0].legend(fontsize=9)

# Log-scale x-axis to visualise lognormal shape
axes[1].hist(np.log10(all_patch_sigmas), bins=80, density=True,
             color='steelblue', alpha=0.7, label='log₁₀(patch σ)')
axes[1].set_xlabel('log₁₀(σ)')
axes[1].set_title('Same data on log scale')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

# ── Print GaussianNoise parameters ────────────────────────────────────────────
print('\n' + '='*62)
print('GaussianNoise parameters (paste into augmentations.py)')
print('='*62)
_s  = shape_fit
_lo = loc_fit
_sc = scale_fit
print(f'# Fitted to JWST COSMOS-Web f150w — {len(TILES)} tiles × {N_PER_TILE} patches')
print(f'# crop size = {int(CROP_AREA**0.5)}×{int(CROP_AREA**0.5)} px  |  N_DILATE = {N_DILATE}')
print(f'_s, _loc, _sc = {_s:.7f}, {_lo:.7e}, {_sc:.7e}')
print(f'self.shape_dist = np.array([_s,   _s,   _s  ])')
print(f'self.loc_dist   = np.array([_loc, _loc, _loc])')
print(f'self.scale_dist = np.array([_sc,  _sc,  _sc ])')
print(f'self.sigma_dist = np.log(self.scale_dist)')
print()
print(f'self.noise_ch_min = np.array([{p05:.7e}]*3)   # p05')
print(f'self.noise_ch_max = np.array([{p95:.7e}]*3)   # p95')
print('='*62)
